# Analysis of Results Common
Comparing individual `_best.csv` files against concatenated batch results, followed by TMT Validation.

In [9]:

import pandas as pd
import glob
import os
import numpy as np
import ast
from scipy.stats import pearsonr, zscore

RESULTS_DIR = "../results_common"


## 1. Load Individual `*_best.csv` Files

In [10]:

# Find all protein_best.csv files
best_files = glob.glob(os.path.join(RESULTS_DIR, "*_best.csv"))
print(f"Found {len(best_files)} individual best files.")

dfs = []
for f in best_files:
    try:
        df = pd.read_csv(f)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {f}: {e}")

if dfs:
    df_best_all = pd.concat(dfs, ignore_index=True)
    # Sort for consistency
    df_best_all.sort_values("protein", inplace=True)
    df_best_all.reset_index(drop=True, inplace=True)
    
    print(f"Concatenated Best Shape: {df_best_all.shape}")
    display(df_best_all.head())
else:
    df_best_all = pd.DataFrame()
    print("No best files loaded.")


Found 100 individual best files.
Concatenated Best Shape: (100, 10)


,protein,best_sample,best_abs_error,mse_all_others,params_dict,feat_coeffs_dict,zscore_genes,zscore_mu,zscore_sigma,zscore_eps
0,ACTA2,J82_URINARY_TRACT,0.000799,0.086437,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'ACAD10': -0.0, 'ACTA2': 0.0, 'ACTB': 0.09919...","['ACAD10', 'ACTA2', 'ACTB', 'ACTG1', 'ACTG2', ...","[3.7855888994261058, 2.58618416764031, 11.3348...","[0.587765897619932, 1.9793520481092222, 0.7363...",1.000000e-08
1,ACTB,HT29_LARGE_INTESTINE,0.000575,0.090003,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'ACAD10': -0.0, 'ACADS': -0.00271022921764425...","['ACAD10', 'ACADS', 'ACTA2', 'ACTB', 'ACTG1', ...","[3.78570365006575, 3.0289030740529643, 2.58929...","[0.5876526030404189, 0.9713270354302064, 1.976...",1.000000e-08
2,AIMP2,JHH1_LIVER,0.000338,0.094300,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'AASS': -0.0, 'ABHD14B': 0.0, 'ACADVL': 0.0, ...","['AASS', 'ABHD14B', 'ACADVL', 'ACSF2', 'ACSS3'...","[2.0416214218453, 5.042360479974216, 7.8144263...","[1.4619632178968838, 0.8788375257666862, 0.708...",1.000000e-08
3,AP2B1,MCF7_BREAST,0.000833,0.053339,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'ABI3BP': 0.0193718232506801, 'ABLIM3': 0.004...","['ABI3BP', 'ABLIM3', 'ACTN1', 'ACTN4', 'ADAM9'...","[1.0650574815337976, 2.391842552545114, 6.8420...","[1.4988497697765129, 1.9066685306693292, 1.405...",1.000000e-08
4,CAPZB,OV90_OVARY,0.002605,0.091850,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'AAK1': 0.0, 'AAMDC': 0.0, 'ABHD17C': -0.0, '...","['AAK1', 'AAMDC', 'ABHD17C', 'ABHD8', 'ABLIM1'...","[3.6125970410388835, 4.923525294133446, 3.6822...","[0.8073625577137409, 0.8666992706185752, 1.245...",1.000000e-08


## 2. Load Batch Result Files

In [11]:

# Find batch files (assuming pattern results_batch_*.csv)
batch_files = glob.glob(os.path.join(RESULTS_DIR, "*batch*.csv"))
print(f"Found {len(batch_files)} batch files.")

dfs_batch = []
for f in batch_files:
    try:
        df = pd.read_csv(f)
        dfs_batch.append(df)
    except Exception as e:
        print(f"Error reading {f}: {e}")

if dfs_batch:
    df_batch_concat = pd.concat(dfs_batch, ignore_index=True)
    df_batch_concat.sort_values("protein", inplace=True)
    df_batch_concat.reset_index(drop=True, inplace=True)
    
    print(f"Concatenated Batch Shape: {df_batch_concat.shape}")
    display(df_batch_concat.head())
else:
    df_batch_concat = pd.DataFrame()
    print("No batch files loaded.")


Found 10 batch files.
Concatenated Batch Shape: (100, 10)


,protein,best_sample,best_abs_error,mse_all_others,params_dict,feat_coeffs_dict,zscore_genes,zscore_mu,zscore_sigma,zscore_eps
0,ACTA2,J82_URINARY_TRACT,0.000799,0.086437,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'ACAD10': -0.0, 'ACTA2': 0.0, 'ACTB': 0.09919...","['ACAD10', 'ACTA2', 'ACTB', 'ACTG1', 'ACTG2', ...","[3.7855888994261058, 2.58618416764031, 11.3348...","[0.587765897619932, 1.9793520481092222, 0.7363...",1.000000e-08
1,ACTB,HT29_LARGE_INTESTINE,0.000575,0.090003,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'ACAD10': -0.0, 'ACADS': -0.00271022921764425...","['ACAD10', 'ACADS', 'ACTA2', 'ACTB', 'ACTG1', ...","[3.78570365006575, 3.0289030740529643, 2.58929...","[0.5876526030404189, 0.9713270354302064, 1.976...",1.000000e-08
2,AIMP2,JHH1_LIVER,0.000338,0.094300,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'AASS': -0.0, 'ABHD14B': 0.0, 'ACADVL': 0.0, ...","['AASS', 'ABHD14B', 'ACADVL', 'ACSF2', 'ACSS3'...","[2.0416214218453, 5.042360479974216, 7.8144263...","[1.4619632178968838, 0.8788375257666862, 0.708...",1.000000e-08
3,AP2B1,MCF7_BREAST,0.000833,0.053339,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'ABI3BP': 0.0193718232506801, 'ABLIM3': 0.004...","['ABI3BP', 'ABLIM3', 'ACTN1', 'ACTN4', 'ADAM9'...","[1.0650574815337976, 2.391842552545114, 6.8420...","[1.4988497697765129, 1.9066685306693292, 1.405...",1.000000e-08
4,CAPZB,OV90_OVARY,0.002605,0.091850,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'AAK1': 0.0, 'AAMDC': 0.0, 'ABHD17C': -0.0, '...","['AAK1', 'AAMDC', 'ABHD17C', 'ABHD8', 'ABLIM1'...","[3.6125970410388835, 4.923525294133446, 3.6822...","[0.8073625577137409, 0.8666992706185752, 1.245...",1.000000e-08


## 3. Compare

In [12]:

# Check if they are the same
if not df_best_all.empty and not df_batch_concat.empty:
    # 1. Compare protein lists
    prots_best = set(df_best_all['protein'].unique())
    prots_batch = set(df_batch_concat['protein'].unique())
    
    missing_in_batch = prots_best - prots_batch
    missing_in_best = prots_batch - prots_best
    
    print(f"Proteins in Best: {len(prots_best)}")
    print(f"Proteins in Batch: {len(prots_batch)}")
    
    if missing_in_batch:
        print(f"Proteins in Best but NOT in Batch: {len(missing_in_batch)}")
    if missing_in_best:
        print(f"Proteins in Batch but NOT in Best: {len(missing_in_best)}")
        
    # Check shape equality
    if df_best_all.shape == df_batch_concat.shape:
         try:
            pd.testing.assert_frame_equal(df_best_all, df_batch_concat, check_like=True)
            print("SUCCESS: DataFrames are identical (pd.testing.assert_frame_equal passed).")
         except AssertionError as e:
            print(f"DataFrames are NOT identical: {e}")
    else:
        print("Shapes differ, cannot be identical.")

else:
    print("Cannot compare - one or both dataframes are empty.")


Proteins in Best: 100
Proteins in Batch: 100
SUCCESS: DataFrames are identical (pd.testing.assert_frame_equal passed).


## 4. TMT Data Loading (Pre-Processed with Gene Symbols)

In [13]:

import joblib
import pandas as pd
import os

# --- UPDATED: Load from Validation Folder ---
RNA_PATH = "../validation/TMT_transcriptomics.joblib"
PROT_PATH = "../validation/TMT_proteomics.joblib"

print(f"Loading Validation TMT Data from: {RNA_PATH} and {PROT_PATH}")

rna_df = None
prot_df = None

if os.path.exists(RNA_PATH) and os.path.exists(PROT_PATH):
    try:
        rna_df = joblib.load(RNA_PATH)
        prot_df = joblib.load(PROT_PATH)
        
        # Ensure we have pandas DataFrames
        if not isinstance(rna_df, pd.DataFrame):
            print("Warning: Loaded RNA object is not a DataFrame.")
        if not isinstance(prot_df, pd.DataFrame):
            print("Warning: Loaded Protein object is not a DataFrame.")

        print(f"TMT RNA Shape: {rna_df.shape}")
        print(f"TMT Protein Shape: {prot_df.shape}")
        
        # Display heads if possible (requires IPython display which is avail in notebook)
        try:
            display(rna_df.head())
            display(prot_df.head())
        except:
            print(rna_df.head())
            print(prot_df.head())
        
    except Exception as e:
        print(f"Error loading joblib files: {e}")
else:
    print(f"Validation files not found. Current CWD: {os.getcwd()}")
    if not os.path.exists("../validation"):
        print("Folder ../validation does not exist.")


Loading Validation TMT Data from: ../validation/TMT_transcriptomics.joblib and ../validation/TMT_proteomics.joblib
TMT RNA Shape: (1924, 15001)
TMT Protein Shape: (1924, 6885)


,A1BG,A1CF,A2LD1,A2M,A2ML1,A4GALT,AAAS,AACS,AADAT,AAGAB,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
BRAIN_2021_1,11.942,NaN,13.391,21.515,14.359,13.935,17.915,15.391,16.498,16.998,...,15.307,16.746,16.046,16.371,17.805,NaN,18.028,17.907,17.322,17.947
BRAIN_2021_2,13.370,NaN,14.481,20.523,15.195,12.702,17.412,14.507,15.787,17.885,...,16.008,16.458,16.514,16.285,17.897,NaN,18.140,19.269,17.802,17.957
BRAIN_2021_3,13.056,NaN,15.181,22.874,13.440,15.041,17.377,14.910,16.085,17.882,...,16.538,17.067,15.761,15.482,17.392,NaN,18.172,18.528,17.626,17.746
BRAIN_2021_4,14.537,NaN,15.114,22.091,14.503,14.747,17.714,15.304,17.188,18.098,...,15.595,16.140,16.180,16.132,17.742,NaN,18.090,18.644,17.952,17.941
BRAIN_2021_5,12.671,NaN,13.666,21.372,14.370,15.341,17.604,15.414,16.392,17.923,...,15.453,16.192,15.955,15.619,17.543,NaN,18.019,19.882,17.800,17.609


,A1BG,A2M,AAAS,AACS,AAGAB,AAK1,AAMDC,AAMP,AAR2,AARS1,...,ZNRD2,ZNRF2,ZPR1,ZRANB2,ZSCAN18,ZSWIM8,ZW10,ZYG11B,ZYX,ZZEF1
BRAIN_2021_1,0.078,0.487,-0.254,-0.144,-0.467,-0.090,-0.079,0.330,-0.265,-0.318,...,-0.015,-0.469,-0.038,0.160,0.660,-0.047,-0.106,0.079,-0.785,-0.488
BRAIN_2021_2,-0.146,0.799,0.184,-0.471,0.245,-0.610,0.119,-0.087,0.385,0.134,...,-0.362,0.010,0.182,0.077,-1.272,0.162,-0.213,0.049,0.138,-0.290
BRAIN_2021_3,0.822,1.096,-0.094,-0.106,-0.248,0.014,-0.700,-0.638,0.593,-0.367,...,0.130,-0.437,0.298,-0.795,-0.165,-0.066,-0.307,-0.172,0.012,-0.132
BRAIN_2021_4,-0.065,0.129,0.048,-0.118,0.304,0.322,-0.555,-0.363,0.365,0.595,...,0.652,-0.259,-0.245,-1.056,-0.600,-0.255,0.464,0.178,-0.720,-0.150
BRAIN_2021_5,-0.764,-1.032,-0.217,-0.696,-0.052,0.345,-0.643,0.069,0.347,-0.325,...,-0.272,-0.644,-0.081,0.160,-0.752,-0.093,0.011,0.113,1.005,-0.230


## 5. Check Features & Normalize

In [14]:

if rna_df is not None:
    # 1. Z-Score Normalize RNA Features (Columns)
    print("Normalizing TMT RNA features (z-score)...")
    # Apply zscore per column (axis=0) if rows are samples
    rna_z = rna_df.apply(zscore) 
    
    print("Normalization complete.")
    display(rna_z.head())
    
    # Check coverage
    if not df_best_all.empty:
        example_coeffs = ast.literal_eval(df_best_all.iloc[0]['feat_coeffs_dict'])
        needed = list(example_coeffs.keys())
        present = [g for g in needed if g in rna_z.columns]
        print(f"Example Model ({df_best_all.iloc[0]['protein']}) feature coverage: {len(present)}/{len(needed)}")


Normalizing TMT RNA features (z-score)...
Normalization complete.


,A1BG,A1CF,A2LD1,A2M,A2ML1,A4GALT,AAAS,AACS,AADAT,AAGAB,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
BRAIN_2021_1,NaN,NaN,NaN,-0.218751,NaN,1.210093,1.527951,1.649890,1.972714,1.314825,...,1.612317,1.759842,NaN,1.880158,1.571434,NaN,1.536977,1.358451,1.497486,1.118601
BRAIN_2021_2,NaN,NaN,NaN,-0.227844,NaN,0.977842,1.434533,1.456120,1.832742,1.474513,...,1.747187,1.703812,NaN,1.863018,1.588575,NaN,1.556896,1.601847,1.585842,1.120100
BRAIN_2021_3,NaN,NaN,NaN,-0.206295,NaN,1.418423,1.428033,1.544456,1.891408,1.473973,...,1.849158,1.822293,NaN,1.702971,1.494487,NaN,1.562587,1.469427,1.553445,1.088467
BRAIN_2021_4,NaN,NaN,NaN,-0.213472,NaN,1.363044,1.490621,1.630820,2.108552,1.512860,...,1.667727,1.641945,NaN,1.832523,1.559697,NaN,1.548004,1.490156,1.613454,1.117702
BRAIN_2021_5,NaN,NaN,NaN,-0.220062,NaN,1.474932,1.470192,1.654931,1.951846,1.481354,...,1.640407,1.652062,NaN,1.730277,1.522620,NaN,1.535377,1.711393,1.585474,1.067928


Example Model (ACTA2) feature coverage: 590/644


## 6. Predict TMT Proteins & Evaluate

In [15]:

# --- DEBUG: Feature Overlap Check ---
if rna_df is not None and not df_best_all.empty:
    print("Debugging Feature Coverage...")
    rna_feats = set(rna_df.columns)
    print(f"Total TMT RNA Features: {len(rna_feats)}")
    
    # Check first 5 models
    for idx, row in df_best_all.head(5).iterrows():
        p_name = row['protein']
        coeffs = ast.literal_eval(row['feat_coeffs_dict'])
        needed = set(coeffs.keys())
        
        missing = needed - rna_feats
        present = needed & rna_feats
        
        print(f"Protein {p_name}: Needed {len(needed)}, Present {len(present)}, Missing {len(missing)}")
        if missing:
            print(f"  -> First 3 missing: {list(missing)[:3]}")
else:
    print("Data not loaded.")
# ------------------------------------


Debugging Feature Coverage...
Total TMT RNA Features: 15001
Protein ACTA2: Needed 644, Present 590, Missing 54
  -> First 3 missing: ['CIPC', 'INTS14', 'GSDME']
Protein ACTB: Needed 644, Present 597, Missing 47
  -> First 3 missing: ['CIPC', 'SEPTIN7', 'GSDME']
Protein AIMP2: Needed 644, Present 575, Missing 69
  -> First 3 missing: ['COA6', 'FAM234B', 'SETD9']
Protein AP2B1: Needed 644, Present 601, Missing 43
  -> First 3 missing: ['DENND2B', 'CARMIL2', 'APMAP']
Protein CAPZB: Needed 644, Present 581, Missing 63
  -> First 3 missing: ['SOWAHC', 'MICOS10', 'ABRAXAS1']


In [16]:

results = []
if rna_df is not None and not df_best_all.empty:
    for idx, row in df_best_all.iterrows():
        p_name = row['protein']
        coeffs = ast.literal_eval(row['feat_coeffs_dict'])
        needed_feats = list(coeffs.keys())
        
        # Check target
        target_present = False
        y_true = None
        if p_name in prot_df.columns:
            target_present = True
            y_true = prot_df[p_name]
        
        # Check features
        missing = [f for f in needed_feats if f not in rna_z.columns]
        if missing:
            # print(f"Skipping {p_name}: missing {len(missing)} features")
            continue
            
        # Predict
        weights = np.array([coeffs[f] for f in needed_feats])
        X = rna_z[needed_feats]
        
        y_pred = X.dot(weights)
        
        mse = np.nan
        corr = np.nan
        
        if target_present:
            common = y_pred.index.intersection(y_true.index)
            if len(common) > 2:
                yp = y_pred.loc[common]
                yt = y_true.loc[common]
                
                valid = (~np.isnan(yp)) & (~np.isnan(yt))
                if valid.sum() > 2:
                    mse = np.mean((yp[valid] - yt[valid])**2)
                    corr, _ = pearsonr(yp[valid], yt[valid])
        
        results.append({
            "Protein": p_name,
            "MSE": mse,
            "Correlation": corr,
            "Target_Present": target_present
        })

    results_df = pd.DataFrame(results)
    print(f"Prediction complete for {len(results_df)} proteins.")
    display(results_df.head())
    
    if not results_df.empty:
        print("Average MSE:", results_df['MSE'].mean())
        print("Average Correlation:", results_df['Correlation'].mean())
else:
    print("Cannot run prediction.")


Prediction complete for 0 proteins.


""
